# ETL + EDA para predicción de **Churn**
## Notebook 1 — Flujo paso a paso

Este notebook está orientado a un problema de **clasificación supervisada**: predecir si un cliente hará **churn** (`Yes`) o no (`No`).

La idea es avanzar como en un proyecto profesional, pero sin perder visibilidad del proceso.  
Por eso lo dividimos en capas lógicas dentro del mismo notebook:

- **Extract**: cargar y entender el dataset.
- **Transform**: limpiar, tipar y preparar los datos.
- **Analyze**: análisis univariado, bivariado y multivariado.
- **Prepare**: dejar una base lista para el modelado.

> Meta de este notebook: **entender y preparar los datos**, no entrenar modelos todavía.

## 1. Importación de librerías

Aquí cargamos las herramientas de trabajo.

- **pandas / numpy**: manipulación de datos.
- **matplotlib / seaborn**: visualización.
- **pathlib**: manejo de rutas.
- **warnings**: para silenciar advertencias que estorben la lectura.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## 2. Configuración de rutas

La idea profesional es no dejar rutas escritas a mano por todo lado.  
Centralizamos eso aquí.

In [ ]:
BASE_DIR = Path("..").resolve()
DATA_RAW = BASE_DIR / "data" / "raw" / "Telco-Customer-Churn.csv"
DATA_PROCESSED_DIR = BASE_DIR / "data" / "processed"
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Archivo fuente:", DATA_RAW)
print("Existe:", DATA_RAW.exists())

## 3. Capa **Extract**
### 3.1 Clase extractora

Aunque estamos en notebook, vale la pena acostumbrarse a una estructura por capas.  
Esta clase se encargará solo de leer el archivo.

In [ ]:
class CSVExtractor:
    def __init__(self, file_path: Path):
        self.file_path = file_path

    def extract(self) -> pd.DataFrame:
        df = pd.read_csv(self.file_path)
        return df

### 3.2 Lectura del dataset

In [ ]:
extractor = CSVExtractor(DATA_RAW)
df_raw = extractor.extract()

print(f"Filas: {df_raw.shape[0]}")
print(f"Columnas: {df_raw.shape[1]}")
df_raw.head()

## 4. Inspección inicial

Antes de tocar nada, hay que mirar el estado del dataset:
- tipos de datos
- ejemplos de registros
- valores nulos
- duplicados
- variable objetivo

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe(include="all").T

### 4.1 Verificación de nulos aparentes

In [ ]:
df_raw.isna().sum().sort_values(ascending=False)

### 4.2 Verificación de duplicados

In [ ]:
duplicates = df_raw.duplicated().sum()
print("Duplicados exactos:", duplicates)

### 4.3 Distribución inicial del target

Esto nos deja ver si el problema está balanceado o no.

In [ ]:
target_dist = df_raw["Churn"].value_counts()
target_pct = df_raw["Churn"].value_counts(normalize=True) * 100

display(pd.DataFrame({
    "frecuencia": target_dist,
    "porcentaje": target_pct.round(2)
}))

## 5. Diagnóstico de calidad de datos

En este dataset hay una columna famosa por dar guerra: **`TotalCharges`**.  
A simple vista parece numérica, pero viene como texto y trae espacios vacíos en algunos registros.

In [ ]:
df_raw["TotalCharges"].dtype

In [ ]:
# Convertimos temporalmente para diagnosticar
totalcharges_numeric = pd.to_numeric(df_raw["TotalCharges"], errors="coerce")

print("Valores no convertibles a número:", totalcharges_numeric.isna().sum())

df_raw.loc[totalcharges_numeric.isna(), ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]].head(15)

### Observación

Los valores no convertibles aparecen como espacios vacíos.  
En este dataset suelen corresponder a clientes con `tenure = 0`, es decir, clientes muy nuevos.

Todavía no los corregimos aquí; primero los diagnosticamos bien.

In [ ]:
df_raw.loc[totalcharges_numeric.isna(), ["tenure", "MonthlyCharges"]].describe()

## 6. Capa **Transform**
### 6.1 Clase transformadora

Aquí concentramos la lógica de limpieza y preparación inicial.

In [ ]:
class TelcoTransformer:
    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()

    def normalize_columns(self) -> pd.DataFrame:
        self.df.columns = (
            self.df.columns
            .str.strip()
            .str.replace(r"(?<!^)(?=[A-Z])", "_", regex=True)
            .str.lower()
        )
        return self.df

    def clean_total_charges(self) -> pd.DataFrame:
        self.df["total_charges"] = pd.to_numeric(self.df["total_charges"], errors="coerce")
        return self.df

    def fix_missing_total_charges(self) -> pd.DataFrame:
        # Estrategia razonable para este dataset:
        # si tenure == 0, total_charges debería ser 0
        self.df.loc[
            (self.df["total_charges"].isna()) & (self.df["tenure"] == 0),
            "total_charges"
        ] = 0
        return self.df

    def encode_target(self) -> pd.DataFrame:
        self.df["churn_flag"] = self.df["churn"].map({"No": 0, "Yes": 1})
        return self.df

    def convert_binary_columns(self) -> pd.DataFrame:
        binary_map = {"No": 0, "Yes": 1}
        cols = ["partner", "dependents", "phone_service", "paperless_billing"]
        for col in cols:
            self.df[f"{col}_flag"] = self.df[col].map(binary_map)
        return self.df

    def adjust_senior_citizen(self) -> pd.DataFrame:
        self.df["senior_citizen"] = self.df["senior_citizen"].astype(int)
        return self.df

    def create_features(self) -> pd.DataFrame:
        self.df["tenure_group"] = pd.cut(
            self.df["tenure"],
            bins=[-1, 12, 24, 48, 72],
            labels=["0-12", "13-24", "25-48", "49-72"]
        )

        self.df["monthly_charge_group"] = pd.cut(
            self.df["monthly_charges"],
            bins=[0, 35, 70, 100, np.inf],
            labels=["Bajo", "Medio", "Alto", "Muy alto"],
            include_lowest=True
        )

        self.df["is_new_customer"] = np.where(self.df["tenure"] <= 12, 1, 0)
        return self.df

    def drop_irrelevant_columns(self) -> pd.DataFrame:
        # customer_id suele eliminarse para ML porque no aporta señal predictiva útil
        return self.df

    def transform(self) -> pd.DataFrame:
        (
            self.normalize_columns()
                .pipe(lambda x: self.clean_total_charges())
                .pipe(lambda x: self.fix_missing_total_charges())
                .pipe(lambda x: self.encode_target())
                .pipe(lambda x: self.convert_binary_columns())
                .pipe(lambda x: self.adjust_senior_citizen())
                .pipe(lambda x: self.create_features())
        )
        return self.df

### 6.2 Aplicación de la transformación

In [ ]:
transformer = TelcoTransformer(df_raw)
df = transformer.transform()

print(df.shape)
df.head()

### 6.3 Verificación posterior a la limpieza

In [ ]:
df.isna().sum().sort_values(ascending=False).head(15)

In [ ]:
print("Duplicados exactos después de transformar:", df.duplicated().sum())
print("Valores nulos en total_charges:", df["total_charges"].isna().sum())

## 7. Análisis univariado

Aquí analizamos **una variable a la vez**.

Lo separamos en:
- variables numéricas
- variables categóricas

### 7.1 Variables numéricas

In [ ]:
numeric_cols = ["tenure", "monthly_charges", "total_charges"]

df[numeric_cols].describe().T

In [ ]:
for col in numeric_cols:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    sns.histplot(df[col], kde=True, ax=axes[0])
    axes[0].set_title(f"Histograma de {col}")

    sns.boxplot(x=df[col], ax=axes[1])
    axes[1].set_title(f"Boxplot de {col}")

    plt.tight_layout()
    plt.show()

### 7.2 Variables categóricas

Revisamos frecuencias y porcentajes.  
Esto sirve para detectar categorías dominantes y entender mejor la población.

In [ ]:
categorical_cols = [
    "gender", "partner", "dependents", "phone_service", "multiple_lines",
    "internet_service", "online_security", "online_backup", "device_protection",
    "tech_support", "streaming_tv", "streaming_movies", "contract",
    "paperless_billing", "payment_method", "churn"
]

for col in categorical_cols:
    print(f"\n===== {col.upper()} =====")
    display(pd.DataFrame({
        "frecuencia": df[col].value_counts(dropna=False),
        "porcentaje": (df[col].value_counts(normalize=True, dropna=False) * 100).round(2)
    }))

In [ ]:
# Algunos gráficos categóricos clave
plot_cols = ["contract", "internet_service", "payment_method", "churn"]

for col in plot_cols:
    plt.figure(figsize=(8, 4))
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order)
    plt.title(f"Distribución de {col}")
    plt.xticks(rotation=30)
    plt.show()

## 8. Análisis bivariado
### Relación de variables con el target `churn`

Aquí ya empezamos a preguntar:
**¿qué variables parecen asociarse con la fuga de clientes?**

### 8.1 Variables categóricas vs churn

In [ ]:
cat_vs_target = [
    "gender", "senior_citizen", "partner", "dependents", "phone_service",
    "multiple_lines", "internet_service", "online_security", "online_backup",
    "device_protection", "tech_support", "streaming_tv", "streaming_movies",
    "contract", "paperless_billing", "payment_method", "tenure_group",
    "monthly_charge_group"
]

for col in cat_vs_target:
    churn_rate = (
        df.groupby(col)["churn_flag"]
          .mean()
          .sort_values(ascending=False)
          .mul(100)
          .round(2)
          .reset_index(name="churn_rate_pct")
    )

    display(churn_rate)

    plt.figure(figsize=(9, 4))
    sns.barplot(data=churn_rate, x=col, y="churn_rate_pct")
    plt.title(f"Tasa de churn por {col}")
    plt.xticks(rotation=35)
    plt.ylabel("Churn (%)")
    plt.show()

### 8.2 Variables numéricas vs churn

Con variables numéricas conviene comparar distribuciones y estadísticas por grupo.

In [ ]:
df.groupby("churn")[numeric_cols].agg(["mean", "median", "std", "min", "max"]).round(2)

In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df, x="churn", y=col)
    plt.title(f"{col} según churn")
    plt.show()

In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(8, 4))
    sns.kdeplot(data=df, x=col, hue="churn", fill=True, common_norm=False, alpha=0.4)
    plt.title(f"Distribución de {col} por churn")
    plt.show()

## 9. Análisis multivariado

Aquí exploramos relaciones entre varias variables al mismo tiempo.

### 9.1 Correlación entre variables numéricas y banderas binarias

In [ ]:
corr_cols = [
    "senior_citizen", "tenure", "monthly_charges", "total_charges",
    "partner_flag", "dependents_flag", "phone_service_flag",
    "paperless_billing_flag", "is_new_customer", "churn_flag"
]

corr_matrix = df[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(10, 7))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matriz de correlación")
plt.show()

### 9.2 Cruce multivariado: contrato + internet + churn

In [ ]:
contract_internet_churn = (
    df.groupby(["contract", "internet_service"])["churn_flag"]
      .mean()
      .mul(100)
      .round(2)
      .reset_index(name="churn_rate_pct")
)

contract_internet_churn

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(
    data=contract_internet_churn,
    x="contract",
    y="churn_rate_pct",
    hue="internet_service"
)
plt.title("Churn por tipo de contrato e internet")
plt.ylabel("Churn (%)")
plt.show()

### 9.3 Cruce multivariado: tenure, monthly charges y churn

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df.sample(min(1500, len(df)), random_state=42),
    x="tenure",
    y="monthly_charges",
    hue="churn",
    alpha=0.7
)
plt.title("Tenure vs Monthly Charges, coloreado por churn")
plt.show()

### 9.4 Tabla dinámica de churn por grupos

In [ ]:
pivot_churn = pd.pivot_table(
    df,
    index="tenure_group",
    columns="contract",
    values="churn_flag",
    aggfunc="mean"
).round(3)

pivot_churn

In [ ]:
plt.figure(figsize=(8, 5))
sns.heatmap(pivot_churn * 100, annot=True, cmap="YlOrRd", fmt=".1f")
plt.title("Heatmap: churn (%) por tenure_group y contract")
plt.show()

## 10. Hallazgos preliminares

Este bloque no calcula nada nuevo: sirve para que dejes tus conclusiones como analista.

Ejemplos de cosas que normalmente aparecen en este dataset:
- Los contratos **mes a mes** suelen tener más churn.
- Menor `tenure` suele asociarse con mayor probabilidad de fuga.
- Clientes con cargos mensuales altos y menor permanencia suelen ser más inestables.
- `total_charges` depende bastante de `tenure`, así que no debe interpretarse aislado.
- `customer_id` no aporta valor predictivo directo y probablemente se elimine en la etapa de modelado.

> Aquí conviene escribir tus hallazgos con base en lo que veas en las gráficas.

## 11. Capa **Prepare**
### Dataset listo para modelado

Todavía no hacemos encoding completo ni train/test split; eso va en el Notebook 2.  
Pero sí dejamos una versión limpia para usar después.

In [ ]:
df_model_base = df.copy()

# En esta etapa solemos retirar el identificador
df_model_base = df_model_base.drop(columns=["customer_id"])

print(df_model_base.shape)
df_model_base.head()

In [ ]:
output_file = DATA_PROCESSED_DIR / "telco_churn_model_base.csv"
df_model_base.to_csv(output_file, index=False)

print("Archivo guardado en:")
print(output_file)

## 12. Próximo paso

En el siguiente notebook haremos:

1. selección de variables
2. encoding de categóricas
3. separación de `X` e `y`
4. train/test split
5. pipelines de preprocesamiento
6. entrenamiento de modelos supervisados de clasificación
7. comparación de métricas

Ese ya es el salto de **análisis** a **machine learning**.